# Directional-Corridor Transformer-PPO v2

This notebook starts a fresh single-incident v2 run. It never resumes or overwrites the rejected v1 experiment.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPOSITORY = Path("/content/G11project")
REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"
if not (REPOSITORY / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(["git", "-C", str(REPOSITORY), "pull", "--ff-only"], check=True)
os.chdir(REPOSITORY)
subprocess.run(["python", "-m", "pip", "install", "-q", "sumo"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "./BackEnd[dev]"], check=True)
sumo_binary = Path(shutil.which("sumo")).resolve()
os.environ["SUMO_HOME"] = str(sumo_binary.parent.parent)
subprocess.run(["python", "BackEnd/scripts/verify_sumo.py"], check=True)
print("Repository:", REPOSITORY)
print("SUMO_HOME:", os.environ["SUMO_HOME"])

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before training"
print("GPU:", torch.cuda.get_device_name(0))
PERSISTENT_ROOT = Path("/content/drive/MyDrive/G11project-directional-corridor-v2")
WORK_ROOT = Path("/content/g11-directional-v2-work")
PIPELINE = Path("BackEnd/scripts/run_adaptive_demo_pipeline.py")
CONFIG = Path("configs/adaptive_demo_training.yaml")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent v2 output:", PERSISTENT_ROOT)

In [ ]:
import json

base_command = [
    "python",
    str(PIPELINE),
    "--persistent-root",
    str(PERSISTENT_ROOT),
    "--work-root",
    str(WORK_ROOT),
    "--config",
    str(CONFIG),
    "--time-budget-minutes",
    "270",
]
while True:
    status_run = subprocess.run(
        base_command + ["--status"], check=True, capture_output=True, text=True
    )
    status = json.loads(status_run.stdout)
    print(json.dumps(status, indent=2, ensure_ascii=False))
    stage = status["next_stage"]
    if stage is None:
        break
    print("Starting/resuming stage:", stage)
    result = subprocess.run(base_command + ["--stage", stage])
    if result.returncode == 75:
        print("Stage paused safely; run this cell again to resume.")
        break
    if result.returncode != 0:
        raise RuntimeError(f"v2 stage {stage} failed with return code {result.returncode}")

In [ ]:
summary_path = PERSISTENT_ROOT / "comparison_results/summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Acceptance:")
    print(json.dumps(summary.get("acceptance"), indent=2, ensure_ascii=False))
    print("Directional behavior:")
    print(json.dumps(summary.get("behavioral_gate"), indent=2, ensure_ascii=False))
manifest = PERSISTENT_ROOT / "champion/model_manifest.json"
print("Presentation manifest ready:", manifest.is_file())